# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [1]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

HypEx 1.0.7 · pandas 2.2.3


### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [2]:
VARIANT = 21   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [3]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

вариант 21: «Второй месяц Расширенного за 1 ₽»
6000 клиентов · test 2978 · control 3022


### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [4]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

клиентов на группу: 3000, σ оборота: 3306 ₽, ρ двух прошлых периодов: 0.614
MDE без снижения дисперсии: 239 ₽
MDE с CUPED:                189 ₽
+3 % к среднему обороту:    98 ₽


#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.

| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | Акция «Второй месяц Расширенного за 1 ₽» повысит средний оборот в партнёрских сервисах на подписчика не меньше чем на 98 ₽ (+3 % к среднему 3 270 ₽) за 4 недели: дешёвый второй месяц удерживает клиента и подталкивает пользоваться партнёрскими сервисами |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | Подписка (её владелец `client_id`), а не отдельный человек: подпиской делятся до двух участников (`members`), и при рандомизации по людям участники одной подписки попали бы в разные группы, видя один и тот же тариф, — эффект размазался бы |
| группы и доли | `control` и `test` по 50 %; стратификация по `acquisition_channel` (пропуск — отдельная страта «без канала»), чтобы каналы привлечения были сбалансированы |
| основная метрика | Оборот в партнёрских сервисах за 4 недели эксперимента, ₽ на клиента (`spend`) |
| метрики-стражи: что не должно ухудшиться | Отток из подписки (`churned`) и число обращений в поддержку (`tickets`); покупки (`purchases`) — вспомогательная метрика |
| α, мощность, MDE — числами из ячейки выше | α = 0,05, мощность 80 %, 3 000 клиентов на группу: MDE без CUPED 239 ₽, с CUPED по `spend_prev` — 189 ₽. Гипотетические +3 % (98 ₽) эксперимент заметить не способен |
| поправка на множественные сравнения | Поправка Холма на 4 метрики (оборот, покупки, отток, обращения) |
| оценщик | CUPED с ковариатой `spend_prev` (θ ≈ 0,64) и разность средних с интервалом Уэлча; бутстреп — как проверка |
| окно | 4 недели эксперимента (31.08–27.09.2026); предпериод 4 недели (03.08–30.08.2026) — только для CUPED; без подглядывания в промежуточные результаты |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | Раскатываем, если эффект после CUPED положителен, его нижняя граница 95 % интервала выше 0 и ни один страж не ухудшился значимо. Дорабатываем и повторяем, если эффект положителен, но страж сработал или интервал не исключает MDE. Отказываемся, если эффект ≤ 0 или ухудшение стражей съедает прибыль |

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [5]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

{'control': 3022, 'test': 2978}
scipy:    p = 0.5700
pilot.py: p = 0.5700


0.57

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [6]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

spend_prev: test 3267.18 ₽, control 3273.54 ₽
p = 0.9406


0.941

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [7]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

test 3654.62 ₽, control 3366.22 ₽
разность 288.40 ₽, интервал 111.00 … 465.80, p = 0.0014


[288.4, 111.0, 465.8]

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [8]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

feature group  control mean   test mean  difference  difference % TTest pass  TTest p-value UTest pass  UTest p-value
  spend  test   3366.215659 3654.616236  288.400577      8.567502         OK       0.001448         OK       0.000515


[0.0014, 0.0005]

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [9]:
# Числа для ответа на вопрос 1 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
print("SRM p:", round(float(stats.chisquare(mine["grp"].value_counts().sort_index()).pvalue), 3))
print("баланс spend_prev p:", round(float(stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False).pvalue), 3))
print("эффект:", round(float(effect["diff"]), 2),
      "интервал:", round(float(effect["lo"]), 2), "…", round(float(effect["hi"]), 2))
print("MDE без CUPED:", round(float(mde_raw), 1), "· MDE с CUPED:", round(float(mde_cuped), 1))
print("t/U p:", round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4))

SRM p: 0.57
баланс spend_prev p: 0.941
эффект: 288.4 интервал: 111.0 … 465.8
MDE без CUPED: 239.2 · MDE с CUPED: 188.8
t/U p: 0.0014 0.0005


#### ✍️ Ответ на вопрос 1

Разбиение можно считать корректным, т.к. SRM даёт p = 0.57 при размерах 2 978 и 3 022, а оборот прошлых четырёх недель в группах совпадал (3 267 против 3 274 руб, p = 0.94). Эффект акции +288.4 руб/клиент, 95 % интервал 111.0 ... 465.8 руб: ноль исключается, но нижняя граница ниже MDE 188.8 руб, поэтому утверждать, что эффект не меньше MDE, данные не позволяют. t-критерий (p = 0.0014) и U-критерий (p = 0.0005) дали один вывод. Если бы они разошлись, причина была бы в тяжёлом хвосте оборота, потому что U-критерий сравнивает распределения и устойчив к отдельным очень крупным тратам, а среднее на них реагирует.

## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [10]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table).pvalue

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

         control    test    diff      lo      hi       p
metric                                                  
churned   0.0344  0.0460  0.0116  0.0016  0.0215  0.0224
tickets   0.0572  0.0567 -0.0005 -0.0126  0.0116  0.9356
churned     0    1
grp               
control  2918  104
test     2841  137
χ² по оттоку: p = 0.0264


[1.16, 0.026, -0.0, 0.936]

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [11]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

HypEx о поправке: There was less than three groups or multitest method wasn't provided
  feature  control mean  test mean  difference  TTest p-value   holm  rejected
    spend     3366.2157  3654.6162    288.4006         0.0014 0.0043      True
purchases        5.0235     5.4788      0.4554         0.0001 0.0003      True
  churned        0.0344     0.0460      0.0116         0.0224 0.0448      True
  tickets        0.0572     0.0567     -0.0005         0.9356 0.9356     False


['churned', 'purchases', 'spend']

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [12]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

theta = 0.641, дисперсия ниже на 36.5%
разность 292.48 ₽, интервал 151.16 … 433.80


[0.641, 36.5, 292.48, 151.16, 433.8]

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [13]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

target covariate variance_reduction
 spend spend_lag              36.5%
    feature group  control mean   test mean  difference  difference % TTest pass  TTest p-value
      spend  test   3366.215659 3654.616236  288.400577      8.567502         OK       0.001448
spend_cuped  test   3364.192174 3656.669617  292.477443      8.693839         OK       0.000050


292.48

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [15]:
# Числа для ответа на вопрос 2 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
print(guards.round(4).to_string())
print("χ² оттока p:", round(float(p_churn), 4))
print("после Холма:", metrics[["feature", "holm"]].round(4).to_dict("records"))
print("ширина интервала: обычный", round(float(effect["hi"] - effect["lo"]), 1),
      "· CUPED", round(float(effect_cuped["hi"] - effect_cuped["lo"]), 1),
      "· сужение, %", round(float((1 - (effect_cuped["hi"] - effect_cuped["lo"]) / (effect["hi"] - effect["lo"])) * 100), 1))
print("оценка эффекта:", round(float(effect["diff"]), 2), "→", round(float(effect_cuped["diff"]), 2))
print("spend_prev: test", round(float(test["spend_prev"].mean()), 1),
      "· control", round(float(control["spend_prev"].mean()), 1))

         control    test    diff      lo      hi       p
metric                                                  
churned   0.0344  0.0460  0.0116  0.0016  0.0215  0.0224
tickets   0.0572  0.0567 -0.0005 -0.0126  0.0116  0.9356
χ² оттока p: 0.0264
после Холма: [{'feature': 'spend', 'holm': 0.0043}, {'feature': 'purchases', 'holm': 0.0003}, {'feature': 'churned', 'holm': 0.0448}, {'feature': 'tickets', 'holm': 0.9356}]
ширина интервала: обычный 354.8 · CUPED 282.6 · сужение, % 20.3
оценка эффекта: 288.4 → 292.48
spend_prev: test 3267.2 · control 3273.5


#### ✍️ Ответ на вопрос 2

Стражи: отток в тесте вырос на 1.16 п.п. (3.44 % → 4.60 %, χ² p = 0.026) - страж сработал, обращения не изменились (p = 0.94). После поправки Холма значимы оборот, покупки и отток (holm p = 0.004; 0.0003; 0.045), обращения — нет. CUPED сузил интервал на 20.3 % (354.8 → 282.6 руб), а оценка почти не изменилась — 288.4 → 292.5 руб: в прошлом периоде тест тратил на 6.4 ₽ меньше контроля (3 267 против 3 274 руб), и поправка это учла.

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [16]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.
rng = np.random.default_rng(VARIANT)
x_test = test["spend"].to_numpy()
x_control = control["spend"].to_numpy()

diff_means = np.array([
    rng.choice(x_test, size=len(x_test)).mean() - rng.choice(x_control, size=len(x_control)).mean()
    for _ in range(2000)
])
lo, hi = np.percentile(diff_means, [2.5, 97.5])

print(f"бутстреп: {lo:.2f} … {hi:.2f} ₽, медиана {np.median(diff_means):.2f} ₽")
answer_9 = [round(float(lo), 2), round(float(hi), 2)]
answer_9

бутстреп: 114.75 … 473.26 ₽, медиана 286.29 ₽


[114.75, 473.26]

### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [17]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.
candidates = {
    "spend_prev": mine["spend_prev"],
    "spend_prev2": mine["spend_prev2"],
    "spend_prev+spend_prev2": mine["spend_prev"] + mine["spend_prev2"],
    "purchases_prev": mine["purchases_prev"],
    "sessions_prev": mine["sessions_prev"],
    "sessions_prev2": mine["sessions_prev2"],
}
r2 = {name: mine["spend"].corr(values) ** 2 * 100 for name, values in candidates.items()}
best = max(r2, key=r2.get)

print({name: round(float(v), 1) for name, v in r2.items()})
print("purchases (после начала пилота):", round(float(mine["spend"].corr(mine["purchases"]) ** 2 * 100), 1))
answer_10 = [best, round(float(r2[best]), 1)]
answer_10

{'spend_prev': 36.5, 'spend_prev2': 33.2, 'spend_prev+spend_prev2': 43.1, 'purchases_prev': 29.0, 'sessions_prev': 38.9, 'sessions_prev2': 37.5}
purchases (после начала пилота): 71.4


['spend_prev+spend_prev2', 43.1]

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [18]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.
mine["t"] = (mine["grp"] == "test").astype(int)
match_feats = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]

res_match = Matching(n_neighbors=5).execute(Dataset(
    roles={"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
           **{f: FeatureRole() for f in match_feats}},
    data=mine, default_role=InfoRole(),
))
match = F(res_match.resume)
print(match.round(2).to_string())

ate = match.loc["ATE"]
answer_11 = [round(float(ate["Effect Size"]), 2), round(float(ate["CI Lower"]), 2), round(float(ate["CI Upper"]), 2)]
answer_11

     Effect Size  Standard Error  P-value  CI Lower  CI Upper outcome
ATT       238.02           78.71      0.0     83.74    392.29   spend
ATC       234.25           79.17      0.0     79.07    389.43   spend
ATE       236.12           76.26      0.0     86.65    385.59   spend


[236.12, 86.65, 385.59]

### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [19]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽

commission = effect_cuped["diff"] * COMMISSION
extra_tickets = guards.loc["tickets", "diff"] * TICKET_COST
extra_churn = guards.loc["churned", "diff"] * CHURN_COST
result = commission - extra_tickets - extra_churn

print(f"комиссия с прироста оборота: {commission:+.2f} ₽")
print(f"вклад обращений:             {-extra_tickets:+.2f} ₽")
print(f"вклад оттока:                {-extra_churn:+.2f} ₽")
print(f"итого на клиента за 4 недели: {result:+.2f} ₽")
answer_12 = round(float(result), 2)
answer_12

комиссия с прироста оборота: +4.39 ₽
вклад обращений:             +0.07 ₽
вклад оттока:                -8.69 ₽
итого на клиента за 4 недели: -4.23 ₽


-4.23

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [20]:
# Числа для ответа на вопрос 3 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
print("ширины интервалов: обычный", round(float(effect["hi"] - effect["lo"]), 1),
      "· CUPED", round(float(effect_cuped["hi"] - effect_cuped["lo"]), 1),
      "· бутстреп", round(float(answer_9[1] - answer_9[0]), 1),
      "· мэтчинг", round(float(answer_11[2] - answer_11[1]), 1))
print("оценки: обычная", round(float(effect["diff"]), 2),
      "· CUPED", round(float(effect_cuped["diff"]), 2),
      "· мэтчинг", answer_11[0])
print("ρ² purchases:", round(float(mine["spend"].corr(mine["purchases"]) ** 2 * 100), 1))
print("деньги на клиента:", answer_12)

ширины интервалов: обычный 354.8 · CUPED 282.6 · бутстреп 358.5 · мэтчинг 298.9
оценки: обычная 288.4 · CUPED 292.48 · мэтчинг 236.12
ρ² purchases: 71.4
деньги на клиента: -4.23


#### ✍️ Ответ на вопрос 3

Больше всего доверяю CUPED: его интервал 282.6 рую - самый узкий, ковариата измерена до эксперимента, а оценка совпала с HypEx (292.5 руб). Бутстреп дал почти ту же ширину, что нормальное приближение (358.5 против 354.8 руб) — при n около 3 000 тяжёлый хвост на интервал почти не влияет; мэтчинг шире (298.9 руб) и смещён ниже (236.1 руб), потому что при случайном делении группы уже сопоставимы, а подбор добавляет свой шум. `purchases` ковариатой брать нельзя: ρ² = 71.4 %, но покупки измерены за время эксперимента и на них влияет сама акция — CUPED вычтет часть эффекта. В деньгах акция даёт −4.23 руб на клиента за 4 недели: комиссия +4.39 руб не покрывает рост оттока 8.69 руб; в расчёте нет стоимости самого второго месяца за 1 руб (≈449 руб), LTV ушедших и эффекта за пределами четырёх недель.

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчёт продакту

**Что проверяли:** акцию «Второй месяц Расширенного за 1 ₽» на 6 000 подписчиков, по 50 % в тест и контроль, четыре недели. Основная метрика — оборот в партнёрских сервисах за 4 недели, стражи — отток и обращения в поддержку; до запуска зафиксировали MDE 189 ₽ после CUPED и правило: раскатываем только при значимом росте без ухудшения стражей.

**Что получилось:** оборот вырос на 288 ₽ на клиента (95 % интервал 111…466 ₽), после CUPED — на 292 ₽ (151…434 ₽), это +8,6 %. Отток вырос с 3,44 % до 4,60 % (+1,16 п.п., p = 0,026), обращения не изменились; после поправки Холма значимы оборот, покупки и отток.

**Какие остались вопросы:** интервал не доказывает, что эффект не меньше 189 ₽, а рост оттока на грани значимости — за четыре недели виден только первый его всплеск. В деньгах по нашим допущениям акция даёт −4,23 ₽ на клиента: комиссия 4,39 ₽ не покрывает потерю от ушедших 8,69 ₽, и здесь ещё не учтена стоимость второго месяца за 1 ₽ (около 449 ₽) и LTV.

**Что рекомендуем:** в текущем виде на всех не раскатывать — по решающему правилу это «дорабатываем и повторяем», потому что сработал страж оттока. Стоит изменить оффер так, чтобы он не подталкивал к уходу (например, давать скидку не всем, а сегментам с низким риском оттока), и повторить эксперимент: для уверенного эффекта +3 % нужно около 11 000 клиентов на группу с CUPED против нынешних 3 000.

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [21]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

NameError: name 'format_problem' is not defined

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [22]:
# Исследование А: ваш код. Всё — от mine.
median_sessions = mine["sessions_prev"].median()
active_test = test[test["sessions_prev"] >= median_sessions]
observed = pd.concat([active_test.assign(t=1), control.assign(t=0)], ignore_index=True)
naive = active_test["spend"].mean() - control["spend"].mean()

feats_a = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]


def att(frame, feats):
    res = Matching(n_neighbors=5).execute(Dataset(
        roles={"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
               **{f: FeatureRole() for f in feats}},
        data=frame, default_role=InfoRole(),
    ))
    return F(res.resume).loc["ATT"]


att_full = att(observed, feats_a)
att_short = att(observed, [f for f in feats_a if f != "sessions_prev"])
active_control = control[control["sessions_prev"] >= median_sessions]
reference = diff_ci(active_test["spend"], active_control["spend"])

print("медиана sessions_prev:", median_sessions, "· активных в тесте:", len(active_test))
print("наивная разность:", round(float(naive), 2))
print("ATT со всеми признаками:", round(float(att_full["Effect Size"]), 2),
      f"[{att_full['CI Lower']:.2f}; {att_full['CI Upper']:.2f}]")
print("ATT без sessions_prev:", round(float(att_short["Effect Size"]), 2),
      f"[{att_short['CI Lower']:.2f}; {att_short['CI Upper']:.2f}]")
print("эталон рандомизации (активные тест vs активные контроль):",
      round(float(reference["diff"]), 2), f"[{reference['lo']:.2f}; {reference['hi']:.2f}]")

медиана sessions_prev: 15.0 · активных в тесте: 1521
наивная разность: 1796.06
ATT со всеми признаками: 210.99 [-19.20; 441.17]
ATT без sessions_prev: 725.93 [491.76; 960.10]
эталон рандомизации (активные тест vs активные контроль): 402.0 [115.66; 688.34]


#### ✍️ Исследование А

**Вопрос:** насколько наивное сравнение «увидевших акцию» со всеми остальными завышает эффект и убирает ли это искажение мэтчинг. **Как проверяли:** собрали наблюдательные данные — 1 521 активный клиент теста (`sessions_prev` не ниже медианы 15) против всех 3 022 контроля — и сравнили их наивно, мэтчингом по шести предпериодным признакам и с эталоном по рандомизации.

**Что получилось:** наивная разность +1 796 ₽; мэтчинг по всем признакам даёт ATT +211 ₽ [−19; 441], эталон по рандомизации — +402 ₽ [116; 688]. Если убрать `sessions_prev` — признак, по которому шёл отбор, — ATT вырастает до +726 ₽, а без `spend_prev` — до +1 660 ₽, почти до наивной разности.

**Что это значит:** мэтчинг снимает смещение только по тем признакам, которые ему дали; неучтённый признак отбора оставляет смещение почти целиком, а интервал при этом узкий и уверенный. **Чего расчёт не доказывает:** отсутствия ненаблюдаемого смещения и переносимости на клиентов, не похожих на активных, — эталон здесь известен только потому, что у нас есть рандомизированный тест.

### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [23]:
# Исследование Б: ваш код. Всё — от mine.
import math
from statistics import NormalDist

rng_b = np.random.default_rng(VARIANT)
control_spend = control["spend"].to_numpy()
control_cuped, theta_control = cuped(control["spend"], control["spend_prev"])
control_cuped = control_cuped.to_numpy()
effect_3pct = 0.03 * control["spend"].mean()


def power(values, n, reps=500):
    hits = 0
    for _ in range(reps):
        sample = rng_b.choice(values, size=2 * n)
        first = sample[:n] + effect_3pct
        hits += stats.ttest_ind(first, sample[n:], equal_var=False).pvalue < 0.05
    return hits / reps


z_norm = NormalDist().inv_cdf
rows = []
for n in (1_000, 3_000, 10_000, 20_000, 40_000):
    rows.append({
        "n на группу": n,
        "симуляция без CUPED": round(power(control_spend, n), 2),
        "формула без CUPED": round(float(NormalDist().cdf(
            effect_3pct / (control_spend.std() * math.sqrt(2 / n)) - z_norm(0.975))), 2),
        "симуляция с CUPED": round(power(control_cuped, n), 2),
        "формула с CUPED": round(float(NormalDist().cdf(
            effect_3pct / (control_cuped.std() * math.sqrt(2 / n)) - z_norm(0.975))), 2),
    })

print(f"средний оборот контроля {control['spend'].mean():.0f} ₽, "
      f"эффект +3 % = {effect_3pct:.0f} ₽, θ по контролю {theta_control:.3f}")
print(pd.DataFrame(rows).to_string(index=False))
print("нужно на группу для 80 %, без CUPED:", n_per_group(float(control_spend.std()), float(effect_3pct)))
print("нужно на группу для 80 %, с CUPED:", n_per_group(float(control_cuped.std()), float(effect_3pct)))

средний оборот контроля 3366 ₽, эффект +3 % = 101 ₽, θ по контролю 0.651
 n на группу  симуляция без CUPED  формула без CUPED  симуляция с CUPED  формула с CUPED
        1000                 0.10               0.10               0.14             0.13
        3000                 0.20               0.21               0.26             0.30
       10000                 0.57               0.55               0.76             0.75
       20000                 0.81               0.84               0.96             0.96
       40000                 0.99               0.99               1.00             1.00
нужно на группу для 80 %, без CUPED: 18041
нужно на группу для 80 %, с CUPED: 11223


#### ✍️ Исследование Б

**Вопрос:** сколько клиентов нужно, чтобы заметить рост оборота на 3 %, и не обманывает ли формула мощности на тяжёлом хвосте. **Как проверяли:** взяли оборот контроля (эффекта там нет), для каждого n по 500 раз выбирали 2n значений с возвращением, к первой половине добавляли 101 ₽ (3 % от 3 366 ₽) и считали долю отвержений t-критерием Уэлча; отдельно повторили для CUPED-оборота с θ по контролю.

**Что получилось:** без CUPED мощность 57 % при 10 000 и 81 % при 20 000 на группу, с CUPED — 76 % и 96 %; 80 % достигается примерно при 18 000 и 11 000 на группу (расчёт по формуле мощности: 18 041 и 11 223 клиента на группу). Наш эксперимент с 3 000 на группу имел мощность лишь около 20 % без CUPED и 26 % с ним — он был рассчитан на MDE 189 ₽, а не на +3 %.

**Что это значит:** формула на этих объёмах ошибается на 1–4 п.п. и сходится с симуляцией при больших n — тяжёлый хвост не ломает её, но на малых n она слегка оптимистична. **Чего расчёт не доказывает:** он берёт распределение контроля как истину и предполагает аддитивный однородный эффект — сезонность, вмешательства и неоднородность эффекта по клиентам он не учитывает, поэтому это оценка мощности, а не гарантия.